In [1]:
import torch
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch import nn, optim
from torch.utils.data import random_split
from torch.utils.data import DataLoader

In [32]:
class NN(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size,50)
        self.fc2 = nn.Linear(50,num_classes)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [38]:
BATCH_SIZE = 32
LEARNING_RATE = 0.001
INPUT_SIZE = 784
NUM_CLASSES = 10
epochs = 3
import os
NUM_WORKERS = os.cpu_count()

In [14]:
entire_dataset = datasets.MNIST(
    root = 'data/',
    download = True,
    transform = transforms.ToTensor(),
    train = True,
)
train_ds, val_ds = random_split(entire_dataset, [50000, 10000],)
test_ds = datasets.MNIST(root = 'data/', download = True, transform = transforms.ToTensor(), train = False)
train_dataloader = DataLoader(train_ds,batch_size=BATCH_SIZE,num_workers=NUM_WORKERS,pin_memory = True, shuffle = True)
val_dataloader = DataLoader(val_ds, batch_size = BATCH_SIZE, num_workers = NUM_WORKERS, pin_memory = True, shuffle = True)
test_dataloader = DataLoader(test_ds,  batch_size = BATCH_SIZE, num_workers = NUM_WORKERS, pin_memory = True, shuffle = False)

100%|█████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [12:14<00:00, 13.5kB/s]
100%|█████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 32.2kB/s]
100%|█████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:49<00:00, 33.6kB/s]
100%|█████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 1.05MB/s]


In [40]:
torch.manual_seed(42)
model = NN(input_size = INPUT_SIZE, num_classes = NUM_CLASSES)
model.to(device)
model

NN(
  (fc1): Linear(in_features=784, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=10, bias=True)
)

In [28]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(params = model.parameters(), lr = LEARNING_RATE)

In [42]:
from tqdm.auto import tqdm
for epoch in tqdm(range(epochs)):
    for batch, (data, targets) in enumerate(train_dataloader):
        data = data.to(device)
        targets = targets.to(device)
        data= data.reshape(data.shape[0], -1)

        scores = model(data)
        loss = criterion(scores, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

  0%|          | 0/3 [00:00<?, ?it/s]

In [44]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()
    with torch.no_grad():
        for x,y in loader:
            x = x.to(device)
            y=y.to(device)
            x = x.reshape(x.shape[0],-1)
            scores = model(x)
            _,predictions = scores.max(1)#this returns value, index of where it appeared ...very efficient than this alternative...alternative use acc +=(torch.argmax(torch.softmax(scores, dim= 1), dim=1)==y).sum().item()
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)##len(predictions)
    model.train()
    return num_correct/num_samples

In [48]:
model.to(device)
print(f"Accuracy on training set: {check_accuracy(train_dataloader, model)*100:.2f}")
print(f"Accuracy on validation set: {check_accuracy(val_dataloader, model)*100:.2f}")
print(f"Accuracy on test set: {check_accuracy(test_dataloader, model)*100:.2f}")

Accuracy on training set: 12.67
Accuracy on validation set: 12.81
Accuracy on test set: 13.56


In [49]:
!pip install pytorch-lightning
import pytorch_lightning as pl

   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---

ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\ABDALAH\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\ABDALAH\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\ABDALAH\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\ABDALAH\anaconda3\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ABDALAH\anaconda3\Lib\http\client.py", line 479, in read
    s = self.fp.read(amt)
        ^^^^^^^^^^^^^^^^^
  File "C:\Users\ABDALAH\anaconda3\Lib\socket.py", line 720, in readinto
    return self._sock.recv_into(b)


ModuleNotFoundError: No module named 'pytorch_lightning'

In [61]:
import sys
print(sys.executable)

C:\Users\ABDALAH\anaconda3\python.exe


In [3]:
!C:\Users\ABDALAH\anaconda3\python.exe -m pip install pytorch-lightning


  Using cached pytorch_lightning-2.5.6-py3-none-any.whl.metadata (20 kB)
  Using cached torchmetrics-1.8.2-py3-none-any.whl.metadata (22 kB)
  Using cached lightning_utilities-0.15.2-py3-none-any.whl.metadata (5.7 kB)
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/831.6 kB ? eta -:--:--
   ------------ --------------------------- 262.1/831.6 kB ? eta -:--:--
   ------------ --------------------------- 262.1/831.6 kB ? eta -:--:--
   ------------ --------------------------- 262.1/831.6 kB ? eta -:--:--
   ------------ --------------------------- 262.1/831.6 kB ? eta -:--:--
   ------------ --------------------------- 262.1/831.6 kB ? eta -:--:--
   ------------ --------------------------- 262.1/831.6 kB ? eta -:--:--
   ----------------------- -------------- 524.3/831.6 kB 212

In [5]:
import pytorch_lightning as pl

In [ ]:
class NN(pl.LightningModule):
    def __init__(self,input_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size,50)
        self.fc2 = nn.Linear(50,num_classes)
        self.loss_fn = nn.CrossEntropyLoss()
        self.accuracy = torchmetrics.Accuracy(task = 'Multi-class', num_classes = num_classes)
        self.my_accuracy = MyAccuracy()
        self.F1Score = torchmetrics.F1Score(task = 'Multi-class', num_classes = num_classes)
    def forward(self,x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    def training_step(self,batch,batch_idx):
        loss,scores,y=self._common_step(batch,batch_idx)
        self.log('train loss', loss)
        return loss
    def validation_step(self,batch,batch_idx):
        loss, scores, y = self._common_step(batch,batch_idx)
        self.log('validation loss', loss)
        return loss
    def test_step(self,batch, batch_idx):
        loss, scores, y = self._common_step(batch,batch_idx)
        self.log('test loss', loss)
        return loss
    def _common_step(self, batch,batch_idx):
        x,y = batch
        x = x.reshape(x.shape[0],-1)
        scores = self.forward(x)
        loss = self.loss_fn(scores, y)
        return loss, scores, y
    def prediction_step(self, batch, batch_idx):
        x,y = batch
        x = x.reshape(x.shape[0], -1)
        scores = self.forward(x)
        predictions = torch.argmax(scores, dim= 1)
        return predictions

    def configure_optimizers(self):
        return optim.Adam(self.parameters(),lr=0.001)

In [ ]:
trainer = pl.Trainer(
    accelerator = 'cpu',
    devices = 1,
    min_epoch = 1,
    max_epoch = 3,
    precision = 16
)
trainer.fit(model, train_dataloader, val_dataloader)
trainer.validate(model, val_dataloader)
trainer.test(model, test_dataloader)

In [66]:
from torchmetrics import Metric
class MyAccuracy(Metric):
    def __init__(self):
        super().__init__()
        self.add_state('total', default=torch.tensor(0), dist_reduce_fx="sum")
        self.add_state('correct', default=torch.tensor(0), dist_reduce_fx="sum")
    def update(self, target, preds):
        preds = torch.argmax(preds, dim= 1)
        assert preds.shape == target.shape
        self.correct += torch.sum(preds == target)
        self.total += target.numel()
    def compute(self):
        return self.correct.float() / self.total.float()

ModuleNotFoundError: No module named 'torchmetrics'

In [ ]:
class MnistDataModule(pl.LightningModule):
    def __init__(self, batch_size,num_workers, data_dir):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.data_dir = data_dir
    def prepare_data(self):
        datasets.MNIST(self.data_dir, download = True, train=True)
        datasets.MNIST(self.data_dir, download = True, train = False)
    def setup(self):
        entire_dataset = datasets.MNIST(
            self.data_dir,
            download = False,
            train=True,
        )
        self.train_ds, self.val_ds = random_split(entire_dataset, [50000,10000])
        self.test_ds = datasets.MNIST(
            self.data_dir,
            download = False,
            train = False
        )
        return train_ds, val_ds, test_ds
    def train_dataloaders(self):
        train_dataloader = DataLoader()
        return train_dataloader
    def val_dataloaders(self):
        val_dataloader = DataLoader()
        return val_dataloader
    def test_dataloaders(self):
        test_dataloader = DataLoader()
        return test_dataloader

model = NN()
dm = MnistDataModule()
trainer = pl.Trainer(accelerator="gpu", devices=1, min_epochs=1, max_epochs=3, precision=16)
trainer.fit(model, dm)
trainer.validate(model, dm)
trainer.test(model, dm)

In [1]:
from pathlib import Path
model_path_summary = Path('pytorch_lightning')
model_path_summary.mkdir(parents = True, exist_ok = True)

In [3]:
%%writefile pytorch_lightning/model.py
class NN(pl.LightningModule):
    def __init__(self,input_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size,50)
        self.fc2 = nn.Linear(50,num_classes)
        self.loss_fn = nn.CrossEntropyLoss()
        self.accuracy = torchmetrics.Accuracy(task = 'Multi-class', num_classes = num_classes)
        self.my_accuracy = MyAccuracy()
        self.F1Score = torchmetrics.F1Score(task = 'Multi-class', num_classes = num_classes)
    def forward(self,x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    def training_step(self, batch, batch_idx):
        x, y = batch
        loss, scores, y = self._common_step(batch, batch_idx)
        accuracy = self.accuracy(scores, y)
        f1_score = self.f1_score(scores, y)
        self.log_dict(
            {
                "train_loss": loss,
                "train_accuracy": accuracy,
                "train_f1_score": f1_score,
            },
            on_step=False,
            on_epoch=True,
            prog_bar=True,
        )
        
        if batch_idx % 100 == 0:
            x = x[:8]
            grid = torchvision.utils.make_grid(x.view(-1, 1, 28, 28))
            self.logger.experiment.add_image("mnist_images", grid, self.global_step)

        return {"loss": loss, "scores": scores, "y": y}
    def training_epoch_end(self, outputs):## depreceated to on_train_epoch_end(self):
        scores = torch.cat([x["scores"] for x in outputs])
        y = torch.cat([x["y"] for x in outputs])
        self.log_dict(
            {
                "train_acc": self.accuracy(scores, y),
                "train_f1": self.f1_score(scores, y),
            },
            on_step=False,
            on_epoch=True,
            prog_bar=True,
        )
    def validation_step(self,batch,batch_idx):
        loss, scores, y = self._common_step(batch,batch_idx)
        self.log('validation loss', loss)
        return loss
    def test_step(self,batch, batch_idx):
        loss, scores, y = self._common_step(batch,batch_idx)
        self.log('test loss', loss)
        return loss
    def _common_step(self, batch,batch_idx):
        x,y = batch
        x = x.reshape(x.shape[0],-1)
        scores = self.forward(x)
        loss = self.loss_fn(scores, y)
        return loss, scores, y
    def prediction_step(self, batch, batch_idx):
        x,y = batch
        x = x.reshape(x.shape[0], -1)
        scores = self.forward(x)
        predictions = torch.argmax(scores, dim= 1)
        return predictions

    def configure_optimizers(self):
        return optim.Adam(self.parameters(),lr=0.001)

Writing pytorch_lightning/model.py


In [5]:
%%writefile pytorch_lightning/dataset.py
class MnistDataModule(pl.LightningModule):
    def __init__(self, batch_size,num_workers, data_dir):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.data_dir = data_dir
    def prepare_data(self):
        datasets.MNIST(self.data_dir, download = True, train=True)
        datasets.MNIST(self.data_dir, download = True, train = False)
    def setup(self):
        entire_dataset = datasets.MNIST(
            self.data_dir,
            download = False,
            train=True,
        )
        self.train_ds, self.val_ds = random_split(entire_dataset, [50000,10000])
        self.test_ds = datasets.MNIST(
            self.data_dir,
            download = False,
            train = False
        )
        return train_ds, val_ds, test_ds
    def train_dataloaders(self):
        train_dataloader = DataLoader(self.train_ds, batch_size = self.batch_size, num_workers = self.num_workers, shuffle = True)
        return train_dataloader
    def val_dataloaders(self):
        val_dataloader = DataLoader(self.val_ds, batch_size = self.batch_size, num_workers = self.num_workers, shuffle = True)
        return val_dataloader
    def test_dataloaders(self):
        test_dataloader = DataLoader(self.test_ds, batch_size = self.batch_size, num_workers = self.num_workers, shuffle = False)
        return test_dataloader

Writing pytorch_lightning/dataset.py


In [7]:
%%writefile pytorch_lightning/config.py
BATCH_SIZE = 32
LEARNING_RATE = 0.001
INPUT_SIZE = 784
NUM_CLASSES = 10
epochs = 3
NUM_WORKERS = 4
DATASET = 'dataset/'
ACCELERATOR = 'gpu'
PRECISION = 16
DEVICES = [0,1]

Writing pytorch_lightning/config.py


In [9]:
%%writefile pytorch_lightning/train.py
from model import NN
from dataset import MnistDataModule
import config
import pytorch_lightning as pl
from callbacks import EarlyStopping, MyPrintingCallback
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.profilers import PyTorchProfiler

torch.set_float32_matmul_precision("medium") 
if __name__ == '__main__':
    logger = TensorBoardLogger("tb_logs", name="mnist_model_v0")
    profiler = PyTorchProfiler(
        on_trace_ready=torch.profiler.tensorboard_trace_handler("tb_logs/profiler0"),
        schedule=torch.profiler.schedule(skip_first=10, wait=1, warmup=1, active=20),
    )
    model = NN(input_size = config.INPUT_SIZE, num_classes = config.NUM_CLASSES)
    dm = MnistDataModule(batch_size = config.BATCH_SIZE, num_workers = config.NUM_WORKERS, data_dir = config.DATASET)
    trainer = pl.Trainer(accelerator=config.ACCELERATOR, devices=config.DEVICES, min_epochs=1, max_epochs=3, precision=config.PRECISION, callbacks = [MyPrintingCallback(), EarlyStopping(monitor = 'val_loss')])
    trainer.fit(model, dm)
    trainer.validate(model, dm)
    trainer.test(model, dm)

Writing pytorch_lightning/train.py


In [11]:
%%writefile pytorch_lightning/callbacks.py
from pytorch_lightning.callbacks import EarlyStopping, Callback
class MyPrintingCallback(Callback):
    def __init__(self):
        super().__init__()
        def on_train_start(self, trainer, pl_module):
            print("Starting to train!")

        def on_train_end(self, trainer, pl_module):
            print("Training is done.")

Writing pytorch_lightning/callbacks.py
